# Extract Filter Parameters From Munich CKAN Resources

This notebook scans parsable resources from the Munich Open Data CKAN portal and creates a catalog of possible dataframe/API filter parameters.

The output is intended for training-data generation. It answers questions like:

- which datasets/resources can be parsed into a dataframe?
- which columns can the model use as filter parameters?
- which operators are valid for each column?
- which filters can be pushed to CKAN datastore APIs vs applied locally after parsing CSV/XML/HTML?

Important distinction:

- `datastore_active=True`: filters can usually be pushed to CKAN via `datastore_search(filters=...)`.
- CSV/XML/HTML resources: filters are dataframe parameters for local filtering after the file has been parsed.

In [1]:
from __future__ import annotations

import json
import re
import time
import xml.etree.ElementTree as ET
from collections import Counter
from html.parser import HTMLParser
from pathlib import Path
from typing import Any
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

BASE_URL = "https://opendata.muenchen.de/api/3/action"
USER_AGENT = "smolnalysis-filter-parameter-extractor/0.1"

RAW_DATA_DIR = Path("../training/data/raw")
GENERATED_DATA_DIR = Path("../training/data/generated")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)

MAX_PACKAGES = 100
PAGE_SIZE = 20
SAMPLE_LIMIT = 500
MAX_EXAMPLE_VALUES = 25

PARSABLE_FORMATS = {"csv", "txt", "xml", "rdf", "rss", "atom", "html", "htm"}

BASE_URL

'https://opendata.muenchen.de/api/3/action'

## CKAN and parsing helpers

In [2]:
def ckan_action(action: str, params: dict[str, Any] | None = None, sleep_s: float = 0.0) -> Any:
    params = params or {}
    query = f"?{urlencode(params, doseq=True)}" if params else ""
    url = f"{BASE_URL}/{action}{query}"
    request = Request(url, headers={"User-Agent": USER_AGENT})

    try:
        with urlopen(request, timeout=45) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError({"url": url, "status": exc.code, "body": body[:2000]}) from exc

    if not payload.get("success"):
        raise RuntimeError({"action": action, "params": params, "error": payload.get("error")})

    if sleep_s:
        time.sleep(sleep_s)

    return payload["result"]


def fetch_text(url: str) -> str:
    request = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=45) as response:
        charset = response.headers.get_content_charset() or "utf-8"
        return response.read().decode(charset, errors="replace")


def normalize_format(resource: dict[str, Any]) -> str:
    fmt = (resource.get("format") or "").strip().lower()
    url = (resource.get("url") or "").lower()
    if fmt:
        return fmt
    for suffix in ["csv", "txt", "xml", "rdf", "rss", "atom", "html", "htm"]:
        if url.endswith(f".{suffix}"):
            return suffix
    return ""

In [3]:
class SimpleHTMLTableParser(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self.tables: list[list[list[str]]] = []
        self._table: list[list[str]] | None = None
        self._row: list[str] | None = None
        self._cell: list[str] | None = None
        self._in_cell = False

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        if tag == "table":
            self._table = []
        elif tag == "tr" and self._table is not None:
            self._row = []
        elif tag in {"td", "th"} and self._row is not None:
            self._cell = []
            self._in_cell = True

    def handle_data(self, data: str) -> None:
        if self._in_cell and self._cell is not None:
            text = data.strip()
            if text:
                self._cell.append(text)

    def handle_endtag(self, tag: str) -> None:
        if tag in {"td", "th"} and self._row is not None and self._cell is not None:
            self._row.append(" ".join(self._cell).strip())
            self._cell = None
            self._in_cell = False
        elif tag == "tr" and self._table is not None and self._row is not None:
            if self._row:
                self._table.append(self._row)
            self._row = None
        elif tag == "table" and self._table is not None:
            if self._table:
                self.tables.append(self._table)
            self._table = None


def flatten_xml_element(element: ET.Element) -> dict[str, Any]:
    row: dict[str, Any] = dict(element.attrib)
    for child in list(element):
        key = child.tag.split("}")[-1]
        value = (child.text or "").strip()
        if list(child):
            for nested_key, nested_value in flatten_xml_element(child).items():
                row[f"{key}.{nested_key}"] = nested_value
        elif value:
            row[key] = value
    return row


def parse_html_table(url: str, table_index: int = 0) -> pd.DataFrame:
    parser = SimpleHTMLTableParser()
    parser.feed(fetch_text(url))
    if not parser.tables:
        raise ValueError("No HTML tables found.")
    table = parser.tables[table_index]
    header, rows = table[0], table[1:]
    width = len(header)
    normalized = [row[:width] + [None] * max(0, width - len(row)) for row in rows]
    return pd.DataFrame(normalized, columns=header)


def parse_xml(url: str, limit: int = SAMPLE_LIMIT) -> pd.DataFrame:
    root = ET.fromstring(fetch_text(url))
    rows = []
    for element in root.iter():
        if list(element):
            row = flatten_xml_element(element)
            if row:
                rows.append(row)
        if len(rows) >= limit:
            break
    if not rows:
        rows = [flatten_xml_element(root)]
    return pd.DataFrame(rows).head(limit)

## Resource sampler

For datastore resources, the field metadata comes from CKAN and filters can be server-side. For file resources, the notebook parses a sample into pandas and derives filter parameters from dataframe columns.

In [4]:
def sample_resource(resource: dict[str, Any], limit: int = SAMPLE_LIMIT) -> tuple[pd.DataFrame, dict[str, Any]]:
    metadata = {
        "filter_mode": "local_dataframe",
        "server_filter_supported": False,
        "source": "file",
    }

    if resource.get("datastore_active"):
        result = ckan_action("datastore_search", {"resource_id": resource["id"], "limit": limit})
        metadata.update({
            "filter_mode": "ckan_datastore",
            "server_filter_supported": True,
            "source": "datastore_search",
            "ckan_fields": result.get("fields", []),
        })
        return pd.DataFrame(result.get("records", [])), metadata

    url = resource.get("url")
    fmt = normalize_format(resource)
    if not url:
        raise ValueError("Resource has no URL.")

    if fmt in {"csv", "txt"}:
        return pd.read_csv(url, sep=None, engine="python", nrows=limit), metadata
    if fmt in {"xml", "rdf", "rss", "atom"} or url.lower().endswith(".xml"):
        return parse_xml(url, limit=limit), metadata
    if fmt in {"html", "htm"} or url.lower().endswith((".html", ".htm")):
        return parse_html_table(url).head(limit), metadata

    raise ValueError(f"Unsupported format: {resource.get('format')} url={url}")

## Filter-parameter inference

The extractor emits conservative operators. Numeric/date-like columns get range operators. Low-cardinality text/category columns get equality and membership operators. Longer text columns get `contains`.

In [5]:
def clean_examples(values: pd.Series, max_values: int = MAX_EXAMPLE_VALUES) -> list[Any]:
    clean = values.dropna()
    if clean.empty:
        return []
    counts = Counter(clean.astype(str).tolist())
    return [value for value, _ in counts.most_common(max_values)]


def looks_like_year(series: pd.Series) -> bool:
    numeric = pd.to_numeric(series.dropna(), errors="coerce").dropna()
    if numeric.empty:
        return False
    return numeric.between(1800, 2100).mean() > 0.9


def looks_like_date(series: pd.Series) -> bool:
    sample = series.dropna().astype(str).head(50)
    if sample.empty:
        return False
    date_patterns = [
        r"^\d{4}-\d{1,2}-\d{1,2}",
        r"^\d{1,2}\.\d{1,2}\.\d{4}",
        r"^\d{1,2}/\d{1,2}/\d{4}",
    ]
    matches = sample.apply(lambda value: any(re.search(pattern, value) for pattern in date_patterns))
    return matches.mean() > 0.6


def infer_column_kind(series: pd.Series) -> str:
    if pd.api.types.is_bool_dtype(series):
        return "boolean"
    if pd.api.types.is_numeric_dtype(series):
        return "year" if looks_like_year(series) else "numeric"
    if looks_like_date(series):
        return "date"

    non_null = series.dropna().astype(str)
    if non_null.empty:
        return "unknown"
    unique_ratio = non_null.nunique() / max(len(non_null), 1)
    avg_len = non_null.str.len().mean()
    if unique_ratio <= 0.3 or non_null.nunique() <= 50:
        return "category"
    if avg_len > 30:
        return "text"
    return "string"


def operators_for_kind(kind: str) -> list[str]:
    if kind in {"numeric", "year", "date"}:
        return ["eq", "neq", "gt", "gte", "lt", "lte", "between", "in"]
    if kind in {"category", "boolean", "string"}:
        return ["eq", "neq", "in", "contains"]
    if kind == "text":
        return ["contains", "eq", "neq"]
    return ["eq", "neq"]


def infer_filter_parameters(df: pd.DataFrame, resource_meta: dict[str, Any]) -> list[dict[str, Any]]:
    params = []
    for column in df.columns:
        series = df[column]
        kind = infer_column_kind(series)
        numeric = pd.to_numeric(series.dropna(), errors="coerce") if kind in {"numeric", "year"} else pd.Series(dtype="float64")
        examples = clean_examples(series)
        params.append({
            "column": str(column),
            "dtype": str(series.dtype),
            "kind": kind,
            "operators": operators_for_kind(kind),
            "nullable": bool(series.isna().any()),
            "non_null_count": int(series.notna().sum()),
            "unique_sample_count": int(series.dropna().nunique()),
            "example_values": examples,
            "min": None if numeric.empty else float(numeric.min()),
            "max": None if numeric.empty else float(numeric.max()),
            "filter_mode": resource_meta["filter_mode"],
            "server_filter_supported": resource_meta["server_filter_supported"],
        })
    return params

## Fetch candidate resources

Increase `MAX_PACKAGES` after the first run succeeds. The portal has many resources, and HTML/XML parsing quality varies.

In [6]:
def search_packages(query: str = "", rows: int = PAGE_SIZE, start: int = 0, sleep_s: float = 0.0) -> dict[str, Any]:
    return ckan_action("package_search", {"q": query, "rows": rows, "start": start}, sleep_s=sleep_s)


packages = []
for start in range(0, MAX_PACKAGES, PAGE_SIZE):
    result = search_packages(rows=min(PAGE_SIZE, MAX_PACKAGES - start), start=start, sleep_s=0.1)
    packages.extend(result["results"])

print(f"packages fetched: {len(packages):,}")

candidate_resources = []
for package in packages:
    for resource in package.get("resources", []):
        fmt = normalize_format(resource)
        if resource.get("datastore_active") or fmt in PARSABLE_FORMATS:
            candidate_resources.append({"package": package, "resource": resource, "format": fmt})

print(f"candidate resources: {len(candidate_resources):,}")
pd.DataFrame([
    {
        "package_name": item["package"].get("name"),
        "package_title": item["package"].get("title"),
        "resource_id": item["resource"].get("id"),
        "resource_name": item["resource"].get("name"),
        "format": item["format"],
        "datastore_active": item["resource"].get("datastore_active"),
    }
    for item in candidate_resources
]).head(30)

packages fetched: 100
candidate resources: 349


,package_name,package_title,resource_id,resource_name,format,datastore_active
0,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,True
1,solarpotenzial_globalstrahlung_p_02,Solarpotenzialanalyse 2018 der Landeshauptstadt München,160ad084-a469-4718-9d82-68ee8ccebdf9,Metadaten (XML),xml,False
2,solarpotenzial_globalstrahlung_p_02,Solarpotenzialanalyse 2018 der Landeshauptstadt München,a1e527c2-9a8b-4fc3-982a-7889683106f6,Fachportal Energie,html,False
3,solarpotenzial_globalstrahlung_p_02,Solarpotenzialanalyse 2018 der Landeshauptstadt München,f0242692-a06d-4908-a12d-fb57211c931a,Open Geodata Portal des Geoportals,html,False
4,enp_baublock_grundwasserpotential_einteilig_25832,Grundwasserpotential,04a6c65d-3b01-4623-9e73-ad6babed6d5c,Metadaten (XML),xml,False
5,enp_baublock_grundwasserpotential_einteilig_25832,Grundwasserpotential,4d0e4d13-14e7-4dad-ab4a-388ea587d676,Fachportal Energie,html,False
6,enp_baublock_grundwasserpotential_einteilig_25832,Grundwasserpotential,edde8548-855f-42bb-8822-f3958407e01d,Open Geodata Portal des Geoportals,html,False
7,abfall_sonstiges_opendata,Sonstige Abfallentsorgungsanlagen (Opendata),ed5cf769-d6d1-4bfc-8073-9e742b10223e,Metadaten (XML),xml,False
8,abfall_sonstiges_opendata,Sonstige Abfallentsorgungsanlagen (Opendata),868191bd-c230-4480-8937-c2c0ebbca8b3,WFS (GML),xml,False
9,abfall_sonstiges_opendata,Sonstige Abfallentsorgungsanlagen (Opendata),514acba4-4b57-49a7-9371-7badb8b69038,WFS (CSV),csv,False


## Extract filter parameters

This cell creates one JSON object per parsable resource. Failed resources are kept with `ok=false` so you can inspect parser gaps.

In [7]:
filter_catalog = []

for index, item in enumerate(candidate_resources, start=1):
    package = item["package"]
    resource = item["resource"]
    fmt = item["format"]
    print(f"[{index}/{len(candidate_resources)}] {package.get('name')} / {resource.get('name')} ({fmt})")

    entry = {
        "ok": False,
        "package_id": package.get("id"),
        "package_name": package.get("name"),
        "package_title": package.get("title"),
        "resource_id": resource.get("id"),
        "resource_name": resource.get("name"),
        "resource_format": fmt,
        "resource_url": resource.get("url"),
        "datastore_active": bool(resource.get("datastore_active")),
        "filter_parameters": [],
        "sample_rows": [],
        "error": None,
    }

    try:
        df, resource_meta = sample_resource(resource, limit=SAMPLE_LIMIT)
        if df.empty:
            raise ValueError("Parsed dataframe is empty.")
        entry.update({
            "ok": True,
            "row_sample_count": int(len(df)),
            "column_count": int(len(df.columns)),
            "filter_mode": resource_meta["filter_mode"],
            "server_filter_supported": resource_meta["server_filter_supported"],
            "filter_parameters": infer_filter_parameters(df, resource_meta),
            "sample_rows": df.head(5).where(pd.notna(df.head(5)), None).to_dict(orient="records"),
        })
    except Exception as exc:
        entry["error"] = str(exc)

    filter_catalog.append(entry)
    time.sleep(0.1)

print("done")
print("ok:", sum(1 for entry in filter_catalog if entry["ok"]))
print("failed:", sum(1 for entry in filter_catalog if not entry["ok"]))

[1/349] indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct / Wohndauer München (csv)
[2/349] solarpotenzial_globalstrahlung_p_02 / Metadaten (XML) (xml)
[3/349] solarpotenzial_globalstrahlung_p_02 / Fachportal Energie (html)
[4/349] solarpotenzial_globalstrahlung_p_02 / Open Geodata Portal des Geoportals (html)
[5/349] enp_baublock_grundwasserpotential_einteilig_25832 / Metadaten (XML) (xml)
[6/349] enp_baublock_grundwasserpotential_einteilig_25832 / Fachportal Energie (html)
[7/349] enp_baublock_grundwasserpotential_einteilig_25832 / Open Geodata Portal des Geoportals (html)
[8/349] abfall_sonstiges_opendata / Metadaten (XML) (xml)
[9/349] abfall_sonstiges_opendata / WFS (GML) (xml)
[10/349] abfall_sonstiges_opendata / WFS (CSV) (csv)
[11/349] abfall_sonstiges_opendata / Open Geodata Portal des Geoportals (html)
[12/349] bestand_sbh_opendata / Metadaten (XML) (xml)
[13/349] bestand_sbh_opendata / WFS (GML) (xml)
[14/349] bestand_sbh_opendata / WFS (CSV) (csv)
[15/349] bestand_s

## Inspect extracted parameters

In [8]:
flat_rows = []
for entry in filter_catalog:
    for param in entry.get("filter_parameters", []):
        flat_rows.append({
            "package_name": entry["package_name"],
            "package_title": entry["package_title"],
            "resource_id": entry["resource_id"],
            "resource_name": entry["resource_name"],
            "format": entry["resource_format"],
            "filter_mode": param["filter_mode"],
            "server_filter_supported": param["server_filter_supported"],
            "column": param["column"],
            "dtype": param["dtype"],
            "kind": param["kind"],
            "operators": ",".join(param["operators"]),
            "example_values": json.dumps(param["example_values"], ensure_ascii=False),
            "min": param["min"],
            "max": param["max"],
        })

filter_df = pd.DataFrame(flat_rows)
print(f"filterable columns: {len(filter_df):,}")
filter_df.head(50)

filterable columns: 14,168


,package_name,package_title,resource_id,resource_name,format,filter_mode,server_filter_supported,column,dtype,kind,operators,example_values,min,max
0,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,_id,int64,numeric,"eq,neq,gt,gte,lt,lte,between,in","[""1"", ""2"", ""3"", ""4"", ""5"", ""6"", ""7"", ""8"", ""9"", ""10"", ""11"", ""12"", ""13"", ""14"", ""15"", ""16"", ""17"", ""18"", ""19"", ""20"", ""21"", ""22"", ""23"", ""24"", ""25""]",1.0,500.0
1,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Indikator,object,category,"eq,neq,in,contains","[""Wohndauer München""]",NaN,NaN
2,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Auspragung,object,category,"eq,neq,in,contains","[""deutsch""]",NaN,NaN
3,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Jahr,int64,year,"eq,neq,gt,gte,lt,lte,between,in","[""2025"", ""2024"", ""2023"", ""2022"", ""2021"", ""2020"", ""2019"", ""2018"", ""2017"", ""2016"", ""2015"", ""2014"", ""2013"", ""2012"", ""2011"", ""2010"", ""2009"", ""2008"", ""2007"", ""20...",2006.0,2025.0
4,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Raumbezug,object,category,"eq,neq,in,contains","[""Stadt München"", ""01 Altstadt - Lehel"", ""02 Ludwigsvorstadt - Isarvorstadt"", ""03 Maxvorstadt"", ""04 Schwabing - West"", ""05 Au - Haidhausen"", ""06 Sendling"", ...",NaN,NaN
5,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Indikatorwert,float64,numeric,"eq,neq,gt,gte,lt,lte,between,in","[""23.1"", ""23.0"", ""24.2"", ""22.9"", ""20.7"", ""22.8"", ""24.0"", ""24.4"", ""21.0"", ""21.1"", ""23.3"", ""22.6"", ""24.1"", ""22.4"", ""20.5"", ""22.0"", ""23.9"", ""24.6"", ""26.4"", ""22...",16.2,26.9
6,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Basiswert 1,object,string,"eq,neq,in,contains","[""1004727"", ""25521935"", ""294434"", ""659053"", ""654200"", ""1107220"", ""966967"", ""634556"", ""997497"", ""396287"", ""1643936"", ""919644"", ""1004373"", ""1179938"", ""1590577...",NaN,NaN
7,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Basiswert 2,object,string,"eq,neq,in,contains","[""40448"", ""15600"", ""68300"", ""70566"", ""40829"", ""75052"", ""1119484"", ""15583"", ""37098"", ""38498"", ""52749"", ""49064"", ""29955"", ""42947"", ""19595"", ""75466"", ""36558"", ...",NaN,NaN
8,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Name Basiswert 1,object,category,"eq,neq,in,contains","[""Summe (Wohndauer in München) (deutsch)""]",NaN,NaN
9,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Name Basiswert 2,object,category,"eq,neq,in,contains","[""Hauptwohnsitzbevölkerung (deutsch)""]",NaN,NaN


In [9]:
if not filter_df.empty:
    display(filter_df.groupby(["filter_mode", "kind"]).size().reset_index(name="columns").sort_values("columns", ascending=False))
    display(filter_df[filter_df["server_filter_supported"]].head(30))
else:
    print("No filter parameters extracted yet.")

,filter_mode,kind,columns
6,local_dataframe,category,12222
7,local_dataframe,date,750
8,local_dataframe,numeric,359
9,local_dataframe,string,344
10,local_dataframe,text,339
0,ckan_datastore,category,75
3,ckan_datastore,string,35
2,ckan_datastore,numeric,27
5,ckan_datastore,year,13
1,ckan_datastore,date,2


,package_name,package_title,resource_id,resource_name,format,filter_mode,server_filter_supported,column,dtype,kind,operators,example_values,min,max
0,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,_id,int64,numeric,"eq,neq,gt,gte,lt,lte,between,in","[""1"", ""2"", ""3"", ""4"", ""5"", ""6"", ""7"", ""8"", ""9"", ""10"", ""11"", ""12"", ""13"", ""14"", ""15"", ""16"", ""17"", ""18"", ""19"", ""20"", ""21"", ""22"", ""23"", ""24"", ""25""]",1.00,500.00
1,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Indikator,object,category,"eq,neq,in,contains","[""Wohndauer München""]",NaN,NaN
2,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Auspragung,object,category,"eq,neq,in,contains","[""deutsch""]",NaN,NaN
3,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Jahr,int64,year,"eq,neq,gt,gte,lt,lte,between,in","[""2025"", ""2024"", ""2023"", ""2022"", ""2021"", ""2020"", ""2019"", ""2018"", ""2017"", ""2016"", ""2015"", ""2014"", ""2013"", ""2012"", ""2011"", ""2010"", ""2009"", ""2008"", ""2007"", ""20...",2006.00,2025.00
4,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Raumbezug,object,category,"eq,neq,in,contains","[""Stadt München"", ""01 Altstadt - Lehel"", ""02 Ludwigsvorstadt - Isarvorstadt"", ""03 Maxvorstadt"", ""04 Schwabing - West"", ""05 Au - Haidhausen"", ""06 Sendling"", ...",NaN,NaN
5,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Indikatorwert,float64,numeric,"eq,neq,gt,gte,lt,lte,between,in","[""23.1"", ""23.0"", ""24.2"", ""22.9"", ""20.7"", ""22.8"", ""24.0"", ""24.4"", ""21.0"", ""21.1"", ""23.3"", ""22.6"", ""24.1"", ""22.4"", ""20.5"", ""22.0"", ""23.9"", ""24.6"", ""26.4"", ""22...",16.20,26.90
6,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Basiswert 1,object,string,"eq,neq,in,contains","[""1004727"", ""25521935"", ""294434"", ""659053"", ""654200"", ""1107220"", ""966967"", ""634556"", ""997497"", ""396287"", ""1643936"", ""919644"", ""1004373"", ""1179938"", ""1590577...",NaN,NaN
7,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Basiswert 2,object,string,"eq,neq,in,contains","[""40448"", ""15600"", ""68300"", ""70566"", ""40829"", ""75052"", ""1119484"", ""15583"", ""37098"", ""38498"", ""52749"", ""49064"", ""29955"", ""42947"", ""19595"", ""75466"", ""36558"", ...",NaN,NaN
8,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Name Basiswert 1,object,category,"eq,neq,in,contains","[""Summe (Wohndauer in München) (deutsch)""]",NaN,NaN
9,indikatorenatlas-bevoelkerung-wohndauer-muenchen-83r65mct,Indikatorenatlas: Bevölkerung - Wohndauer München,773a81c6-1467-4c4a-adde-329776f4b44e,Wohndauer München,csv,ckan_datastore,True,Name Basiswert 2,object,category,"eq,neq,in,contains","[""Hauptwohnsitzbevölkerung (deutsch)""]",NaN,NaN


## Save artifacts

The JSONL file is the useful training-data source. The CSV file is for quick manual inspection.

In [10]:
jsonl_path = RAW_DATA_DIR / "munich_filter_parameters.jsonl"
csv_path = GENERATED_DATA_DIR / "munich_filter_parameters.csv"

with jsonl_path.open("w", encoding="utf-8") as file:
    for entry in filter_catalog:
        file.write(json.dumps(entry, ensure_ascii=False) + "\n")

filter_df.to_csv(csv_path, index=False)

jsonl_path, csv_path

(PosixPath('../training/data/raw/munich_filter_parameters.jsonl'),
 PosixPath('../training/data/generated/munich_filter_parameters.csv'))

## How to use this for training examples

Use each `filter_parameters` entry to generate examples for `generate_open_data_query` or `generate_dataframe_query`.

For example, a categorical parameter can create examples like:

```json
{
  "filters": [
    {"column": "geschlecht", "operator": "eq", "value": "w"}
  ],
  "sort": [{"column": "anzahl", "direction": "desc"}],
  "limit": 10
}
```

A numeric or year parameter can create range examples:

```json
{
  "filters": [
    {"column": "jahr", "operator": "between", "value": [2020, 2024]}
  ],
  "limit": 100
}
```

Remember: only datastore-backed resources should be described as API-filterable. CSV/XML/HTML parameters are still useful, but they are local dataframe filters after parsing.